# C1 · Referencia independiente DINOv2-L/14

**Estado:** protocolo fijado antes de calcular sus resultados de test.

**Objetivo y pregunta.** Comparar E4 con un encoder visual independiente: ¿cómo rinde DINOv2-L/14 congelado bajo las mismas consultas y galerías?

**Hipótesis.** Las características auto-supervisadas pueden ser competitivas y revelar cuánto depende el resultado del backbone. No se presupone que este control supere a E4.

**Intervención.** DINOv2-L/14, token CLS normalizado, pesos congelados y revisión inmutable. Se conserva el detector/recorte E2. Se utiliza el procesador oficial fijado (resize 256, centro 224, normalización ImageNet). No se usa cabeza E3 ni LoRA.

**Controles y muestra.** Las mismas 277 fotos públicas del test v1 inmutable y 502 fotos PetTrace v6; galerías separadas por dominio y especie, leave-one-out por fotografía. Los hashes y las identidades se contrastan con el recibo publicado de E4 antes de inferencia. Se descargan solo shards de test de las fuentes preparadas; no hay limpieza ni nuevo split.

**Entrenamiento y selección.** Ninguno. Arquitectura, pesos, representación y preprocesamiento se fijan aquí; no se seleccionan usando métricas de test. El modelo no se registra ni promociona para servir; todos los resultados se publican obligatoriamente en MLflow.

**Métricas y resultados.** Rank-1/2/5/10 y mAP, por dominio y especie; EER descriptiva sobre los mismos pares. Resultados pendientes hasta terminar inferencia y publicación verificable. Los archivos numéricos permiten reintentar publicación sin repetir inferencia.

**Limitaciones.** Backbone y presupuesto de preentrenamiento distintos; esta es una referencia externa, no una ablación de E4. DINOv2 no fue ajustado al corpus de PetTrace. El preentrenamiento externo no permite descartar toda coincidencia con imágenes públicas. La comparación no evalúa separación entre sesiones ni búsqueda de animales ausentes de la galería. No prueba una mejora de despliegue ni una aportación propia.

Método externo: [Oquab et al., DINOv2](https://arxiv.org/abs/2304.07193), [repositorio oficial](https://github.com/facebookresearch/dinov2). Conversión/procesador: [facebook/dinov2-large](https://huggingface.co/facebook/dinov2-large/tree/47b73eefe95e8d44ec3623f8890bd894b6ea2d6c). Recorte: [Ultralytics YOLOv8](https://docs.ultralytics.com/models/yolov8/). Elección del control, protocolo de recuperación y galería son decisiones propias.

### Entorno
Ejecución local con el entorno del repositorio y MPS, sin instalación ni launcher dentro del notebook. Las versiones efectivas se conservan con los resultados. Credenciales existentes desde `.env`, nunca impresas.

In [1]:
import csv, hashlib, importlib.metadata, json, os, platform, shutil, sys, tarfile, time
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
from uuid import uuid4
from dotenv import load_dotenv
import numpy as np
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # CPU interpolation for unsupported MPS operations.
import torch
import mlflow
from huggingface_hub import hf_hub_download, snapshot_download
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.benchmark import Dataset, Photo
from pettrace.data.refined_sources import RELEASES
from pettrace.data.verification import sha256
from pettrace.experiments.e1 import _private_dataset
from pettrace.experiments.zero_shot import save_publication_input, load_publication_input

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Python 3.11–3.13 required")
if not torch.backends.mps.is_available():
    raise RuntimeError("This fixed execution requires the Mac MPS device")
ROOT = Path.cwd()
assert (ROOT / "src/pettrace").is_dir(), "Run from the repository root"
load_dotenv(ROOT / ".env")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
os.environ["MLFLOW_REGISTRY_URI"] = PRODUCTION_URI
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if not os.environ.get(name):
        raise RuntimeError(f"Missing {name}")
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
PUBLIC_RUN = "358cb901f3e34483b24acc6ce93649bb"
MODEL_REPO = "facebook/dinov2-large"
MODEL_REVISION = "47b73eefe95e8d44ec3623f8890bd894b6ea2d6c"
LOG_TO_MLFLOW = True
RUN_LABEL = "20261118T063806Z-cc261f74"  # Fixed study receipt; reruns resume publication.
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
REPORT = ROOT / "reports/c1-dinov2-reference" / RUN_LABEL
REPORT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "reports/c1-dinov2-reference/cache"
CACHE.mkdir(exist_ok=True)
SOURCE = source_identity()
method_cells = json.loads((ROOT / "notebooks/c1_dinov2_reference.ipynb").read_text())["cells"]
METHOD_SHA256 = hashlib.sha256(json.dumps([c["source"] for c in method_cells],sort_keys=True).encode()).hexdigest()
print("Study:", RUN_LABEL, "| Device: MPS | Package:", SOURCE["package_version"])


2026/11/18 01:39:49 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Study: 20261118T063806Z-cc261f74 | Device: MPS | Package: 0.2.42


## Fase 1. Preparación y preprocesamiento de datos
Carga y verificación de snapshots existentes. Se comprueba el inventario completo de etiquetas y solo se materializan las fotos del split test ya fijado. Las imágenes privadas se verifican por snapshot y se mantienen en una galería independiente.

In [2]:
client = mlflow.MlflowClient()
receipt_path = Path(client.download_artifacts(PUBLIC_RUN, "report/dataset-selection.json", str(CACHE)))
receipt = json.loads(receipt_path.read_text())
assert receipt["source"]["commit"] == "a724c1099a5432609c853958e88b68c4cf82d871"
expected = {(r["animal"], r["species"], r["sha256"]) for r in receipt["rows"]}
photos, rows = [], []
for source in ("lcw", "dogs-world"):
    spec = RELEASES[source]
    root = Path(snapshot_download(spec["repo_id"], repo_type="dataset", revision=spec["v2_commit"],
        allow_patterns=["versions/v1/dataset.json", "versions/v1/metadata.csv", "versions/v1/data/test/*"]))
    manifest = json.loads((root / "versions/v1/dataset.json").read_text())
    meta = root / "versions/v1/metadata.csv"
    assert sha256(meta) == manifest["file_sha256"]["versions/v1/metadata.csv"]
    with meta.open(newline="") as f:
        all_rows = list(csv.DictReader(f))
    subset = [r for r in all_rows if r["split"] == "test"]
    assert not {r["identity_id"] for r in subset} & {r["identity_id"] for r in all_rows if r["split"] != "test"}
    by_hash = {r["sha256"]: r for r in subset}
    assert len(by_hash) == len(subset)
    target_dir = CACHE / "images" / source
    target_dir.mkdir(parents=True, exist_ok=True)
    found = set()
    for shard in manifest["shards"]:
        if "/data/test/" not in shard["path"]:
            continue
        assert not Path(shard["path"]).is_absolute() and ".." not in Path(shard["path"]).parts
        archive = root / shard["path"]
        assert sha256(archive) == shard["sha256"]
        with tarfile.open(archive) as tar:
            for member in tar:
                name = Path(member.name)
                assert member.isfile() and name.name == member.name
                if name.suffix == ".json":
                    continue
                assert name.suffix.lower() in {".jpg", ".jpeg", ".png"}
                assert name.stem in by_hash and name.stem not in found
                target = target_dir / name.name
                if not target.exists() or sha256(target) != name.stem:
                    data = tar.extractfile(member).read()
                    assert hashlib.sha256(data).hexdigest() == name.stem
                    target.write_bytes(data)
                row = by_hash[name.stem]
                photos.append(Photo(str(target), row["identity_id"], row["species"], "test", 0.0))
                rows.append((row["identity_id"], row["species"], name.stem))
                found.add(name.stem)
    assert found == set(by_hash)
assert set(rows) == expected and len(rows) == len(expected) == 277
identity = {"prepared_fingerprint": receipt["prepared_fingerprint"], "split": "test", "rows": sorted(rows)}
digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert digest == receipt["evaluation_fingerprint"]
public_ds = Dataset(receipt["source"]["repo"], "v2", receipt["source"]["commit"], digest, str(CACHE),
    {p.animal: {"species":p.species,"split":"test"} for p in photos}, photos)
private_ds = _private_dataset("v6", DATASET_COMMIT, CACHE / "private")
assert len(private_ds.photos) == 502 and private_ds.fingerprint == "6dcae9a47764beb0"
assert not {r[2] for r in rows} & {sha256(p.path) for p in private_ds.photos}
(REPORT / "dataset-selection.json").write_text(json.dumps(receipt, indent=2))
print("Verified:", len(public_ds.photos), "public photos /", len(public_ds.animals), "identities;",
      len(private_ds.photos), "private photos /", len(private_ds.animals), "identities")


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 506 files:   0%|          | 0/506 [00:00<?, ?it/s]

Verified: 277 public photos / 60 identities; 502 private photos / 50 identities


## Fase 2. Representación y entrenamiento
No se ajusta ningún peso. Se obtiene CLS normalizado y se verifica paridad CPU/MPS sobre las doce imágenes sintéticas de control del proyecto. Esta verificación es numérica; no selecciona un modelo con datos de test. Se registran los hashes de pesos, configuración y detector.

Compatibilidad: la versión local de PyTorch no implementa la interpolación bicúbica en MPS. Se habilita el fallback oficial a CPU para esa operación; los pesos y el procesador no cambian. Se conserva el error inicial en `preflight-mps-unsupported.ipynb` y se exige igualmente paridad numérica antes del test.


In [3]:
from transformers import AutoImageProcessor, AutoModel
from pettrace.evaluation.core import load_model
from pettrace.models.artifacts import Model
from pettrace.models.golden import golden_images, compare, fingerprint

model_dir = Path(snapshot_download(MODEL_REPO, revision=MODEL_REVISION,
    allow_patterns=["model.safetensors", "config.json", "preprocessor_config.json", "README.md"]))
processor = AutoImageProcessor.from_pretrained(str(model_dir), use_fast=False, trust_remote_code=False)
backbone = AutoModel.from_pretrained(str(model_dir), trust_remote_code=False).eval().requires_grad_(False)
base = load_model("models:/pettrace-embedder/2")
assert base.golden_ok and base.pipeline.detector is not None

class ReferencePipeline:
    def __init__(self, backbone, processor, detector):
        self.backbone, self.processor, self.detector = backbone, processor, detector
    def embed(self, images):
        values = []
        device = next(self.backbone.parameters()).device
        for offset in range(0, len(images), 8):
            crops = [self.detector.crop(image.convert("RGB")) for image in images[offset:offset+8]]
            pixels = self.processor(images=crops, return_tensors="pt")["pixel_values"].to(device)
            with torch.inference_mode():
                features = self.backbone(pixel_values=pixels).last_hidden_state[:, 0]
                values.append(torch.nn.functional.normalize(features, dim=-1).cpu().numpy())
        return np.concatenate(values)

pipeline = ReferencePipeline(backbone, processor, base.pipeline.detector)
golden_cpu = pipeline.embed(golden_images())
backbone.to("mps")
golden_mps = pipeline.embed(golden_images())
ok, minimum_cosine, maximum_error = compare(golden_cpu, golden_mps)
assert ok, (minimum_cosine, maximum_error)
np.save(REPORT / "golden.npy", golden_mps)
parity = {"cpu_mps_min_cosine":minimum_cosine,"cpu_mps_max_abs":maximum_error,"golden_images":12}
(REPORT / "parity.json").write_text(json.dumps(parity,indent=2))
hashes = {name:sha256(model_dir/name) for name in ("model.safetensors","config.json","preprocessor_config.json")}
hashes.update({"detector."+name:value for name,value in base.sha256.items() if "detector" in name})
assert any(name.startswith("detector.") for name in hashes)
model = Model(f"hf://{MODEL_REPO}@{MODEL_REVISION}",str(model_dir),pipeline,
              "dinov2-large-cls-yolo-crop@47b73ee", "", hashes, ok, fingerprint(golden_mps))
design = {"experiment":"C1", "method":"frozen DINOv2-L/14 CLS with E2 crop",
          "model_repo":MODEL_REPO,"model_revision":MODEL_REVISION,"model_sha256":hashes,
          "processor":processor.to_dict(),"detector_spec":base.pipeline.spec,
          "training":None,"selection":None,"test_used_for_selection":False,
          "phases":["data_preparation","representation_and_training","evaluation"],
          "gallery":"closed_set_same_species","query_unit":"photograph","capture_session_disjoint":False,
          "mlflow_publication_required":True,"register_model":False,"parity":parity,
          "notebook_method_sha256":METHOD_SHA256}
(REPORT / "experiment-design.json").write_text(json.dumps(design,indent=2))
print("Frozen model verified:",parity,"| parameters:",sum(p.numel() for p in backbone.parameters()))


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/torch/nn/functional.py:4594: UserWarning: The operator 'aten::upsample_bicubic2d.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  return torch._C._nn.upsample_bicubic2d(


Frozen model verified: {'cpu_mps_min_cosine': 0.9999999403953552, 'cpu_mps_max_abs': 2.44826078414917e-05, 'golden_images': 12} | parameters: 304368640


## Fase 3. Evaluación y análisis
Inferencia en MPS; consultas y galerías idénticas al protocolo publicado. Todas las métricas se conservan y publican. La celda siguiente reutiliza `evaluation-input.json` si una publicación anterior falló, sin recalcular representaciones. No se promueven aliases ni se registra este control externo.

In [4]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save

publications = {}
for domain, dataset in (("public",public_ds),("private",private_ds)):
    folder = REPORT / domain
    folder.mkdir(exist_ok=True)
    for name in ("experiment-design.json","dataset-selection.json","parity.json","golden.npy"):
        shutil.copy2(REPORT/name, folder/name)
    for name in ("README.md","config.json","preprocessor_config.json"):
        shutil.copy2(model_dir/name, folder/name)
    if (folder / "evaluation-input.json").exists():
        ev = load_publication_input(folder)
        ev.dataset = dataset
    else:
        started = time.monotonic()
        ev = evaluate(dataset,pipeline,zero_shot=True)
        elapsed = time.monotonic()-started
        save(ev,model,str(folder))
        save_publication_input(ev,folder)
        runtime = {**SOURCE,"execution_id":RUN_LABEL,"python":platform.python_version(),
                   "platform":platform.platform(),"inference_backend":"torch_mps_with_cpu_fallback","seconds":elapsed,
                   "model_sha256":hashes,"batch":8,"notebook_method_sha256":METHOD_SHA256,
                   "packages":{name:importlib.metadata.version(name) for name in
                               ("torch","transformers","numpy","pillow","huggingface-hub","mlflow-skinny")}}
        (folder / "runtime.json").write_text(json.dumps(runtime,indent=2))
    run_id = publish_evaluation(ev,model,folder,SOURCE["source_sha256"],experiment="C1",
        expected_model_uri=model.uri,expected_model_sha256=hashes,
        expected_dataset_version=dataset.version,expected_protocol="zero-shot")
    assert client.get_run(run_id).info.status == "FINISHED"
    publication = {"published":True,"run_id":run_id,"domain":domain,"study_id":RUN_LABEL}
    (folder/"publication.json").write_text(json.dumps(publication,indent=2))
    client.log_artifact(run_id,str(folder/"publication.json"),artifact_path="report")
    client.set_tag(run_id,"study_id",RUN_LABEL)
    client.set_tag(run_id,"purpose","independent_reference")
    publications[domain] = publication
    print(domain,ev.result["test"]["global"],publication,flush=True)
(REPORT/"publication.json").write_text(json.dumps(publications,indent=2))
print("Complete: both domains published, no model registration or alias promotion.")


🏃 View run C1 · dinov2-large-cls-yolo-crop@47b73ee on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/3e604ebd78254956a2bc3f9a9f9a199c
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9205776173285198, 'rank2': 0.9675090252707581, 'rank5': 0.9963898916967509, 'rank10': 0.9963898916967509, 'mAP': 0.8383383805012403} {'published': True, 'run_id': '3e604ebd78254956a2bc3f9a9f9a199c', 'domain': 'public', 'study_id': '20261118T063806Z-cc261f74'}


🏃 View run C1 · dinov2-large-cls-yolo-crop@47b73ee on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/39e0a0fc04904406893ca2adbac98e6b
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


private {'queries': 502, 'gallery_animals': 33, 'rank1': 0.9482071713147411, 'rank2': 0.9880478087649402, 'rank5': 0.99800796812749, 'rank10': 1.0, 'mAP': 0.8650339304191786} {'published': True, 'run_id': '39e0a0fc04904406893ca2adbac98e6b', 'domain': 'private', 'study_id': '20261118T063806Z-cc261f74'}


Complete: both domains published, no model registration or alias promotion.
